In [8]:
import wandb
import pandas as pd
import numpy as np

# 1. Setup
api = wandb.Api()
entity = "luis-perdigao-instituto-superior-t-cnico"
project = "PCS_ET_v2"
runs = api.runs(f"{entity}/{project}")

print(f"Found {len(runs)} runs. Retrieving Test metrics at Best Validation Epoch...")

data_list = []

for i, run in enumerate(runs):
    if i % 10 == 0: print(f"Processing run {i}/{len(runs)}...")

    # A. Get Config
    config = {k: v for k, v in run.config.items() if not k.startswith('_')}
    
    # B. Fetch History
    # We need validation to find the index, and test metrics to report
    keys = ["accuracy_validation", "accuracy_test", "c_accuracy_test"]
    history = pd.DataFrame(run.scan_history(keys=keys))
    
    # Initialize defaults
    final_rank_acc = np.nan
    final_class_acc = np.nan
    best_val_acc = np.nan
    
    if not history.empty and "accuracy_validation" in history.columns:
        # -------------------------------------------------------
        # CRITICAL LOGIC: Model Selection based on Validation
        # -------------------------------------------------------
        
        # 1. Find the index (epoch) where VALIDATION accuracy was max
        best_epoch_idx = history["accuracy_validation"].idxmax()
        
        # 2. Record the Validation score (just for sanity check)
        best_val_acc = history.loc[best_epoch_idx, "accuracy_validation"]
        
        # 3. Retrieve the TEST scores from that SAME epoch
        # (This corresponds to the weights saved as 'best_model')
        if "accuracy_test" in history.columns:
            final_rank_acc = history.loc[best_epoch_idx, "accuracy_test"]
        
        if "c_accuracy_test" in history.columns:
            final_class_acc = history.loc[best_epoch_idx, "c_accuracy_test"]
            
    else:
        # Fallback (only if history failed, use summary but this is less precise)
        final_rank_acc = run.summary.get("accuracy_test", np.nan)
        final_class_acc = run.summary.get("c_accuracy_test", np.nan)

    # C. Build Row
    data_list.append({
        "run_name": run.name,
        "backbone": config.get("backbone", "unknown"),
        "pooling": config.get("pooling", "unknown"),
        "num_ft_blocks": config.get("num_ft_blocks", 0),
        "selected_val_acc": best_val_acc, # Good to keep for debug
        "test_rank_acc": final_rank_acc,  # The result at best val epoch
        "test_class_acc": final_class_acc # The result at best val epoch
    })

# 3. Save
df = pd.DataFrame(data_list)
df.to_csv("backbone_results.csv", index=False)
print("Done! Saved 'backbone_results.csv'.")

Found 75 runs. Retrieving Test metrics at Best Validation Epoch...
Processing run 0/75...
Processing run 10/75...
Processing run 20/75...
Processing run 30/75...
Processing run 40/75...
Processing run 50/75...
Processing run 60/75...
Processing run 70/75...
Done! Saved 'backbone_results.csv'.


In [11]:
import pandas as pd
import numpy as np

def generate_latex_table_val_selected(csv_path="backbone_results.csv"):
    # 1. Load Data
    df = pd.read_csv(csv_path)

    # 2. Clean Names
    backbone_map = {
        'dinov3_vitb16': 'DINOv3',
        'deit3_base_patch16_224': 'DeiT III',
        'vit_base_patch16_224': 'ViT-Base',
        'vit_base_patch16_clip_224': 'ViT-Base (CLIP)',
        'vit_small': 'ViT-Small'
    }
    pooling_map = {'cls': 'CLS', 'concat': 'Concat', 'topk': 'Top-K'}

    df['backbone'] = df['backbone'].map(backbone_map).fillna(df['backbone'])
    df['pooling'] = df['pooling'].map(pooling_map).fillna(df['pooling'])

    # 3. Pivot tables
    # We use the new columns we just extracted
    pivot_rank = df.pivot_table(index=['backbone', 'pooling'], columns='num_ft_blocks', values='test_rank_acc') * 100
    pivot_class = df.pivot_table(index=['backbone', 'pooling'], columns='num_ft_blocks', values='test_class_acc') * 100

    # 4. Generate LaTeX
    latex_str = []
    latex_str.append("\\begin{table*}[ht]")
    latex_str.append("\\centering")
    latex_str.append("\\small")
    latex_str.append("\\caption{Test Accuracy (Rank / Class) selected at best Validation Epoch.}")
    latex_str.append("\\label{tab:backbone_results}")
    latex_str.append("\\begin{tabular}{llccccc}")
    latex_str.append("\\toprule")
    latex_str.append(" & & \\multicolumn{5}{c}{Unfreeze Depth (Blocks)} \\\\")
    latex_str.append("\\cmidrule(lr){3-7}")
    latex_str.append("Backbone & Pooling & No Finetune & 1 & 4 & 8 & 12 \\\\")
    latex_str.append("\\midrule")

    current_backbone = None
    
    # Sort for consistency
    pivot_rank = pivot_rank.sort_index(level=0)
    pivot_class = pivot_class.sort_index(level=0)

    for (backbone, pooling) in pivot_rank.index:
        row_rank = pivot_rank.loc[(backbone, pooling)]
        row_class = pivot_class.loc[(backbone, pooling)]
        
        # We still highlight the best result in the row to show which model performed best
        max_rank = row_rank.max()
        max_class = row_class.max()

        if backbone != current_backbone:
            latex_str.append("\\midrule")
            backbone_display = f"\\textbf{{{backbone}}}"
            current_backbone = backbone
        else:
            backbone_display = ""
            
        row_values = []
        for col in pivot_rank.columns:
            val_r = row_rank.get(col, np.nan)
            val_c = row_class.get(col, np.nan)
            
            if np.isnan(val_r) or np.isnan(val_c):
                row_values.append("-")
            else:
                s_r = f"{val_r:.2f}"
                if abs(val_r - max_rank) < 1e-9: s_r = f"\\textbf{{{s_r}}}"
                
                s_c = f"{val_c:.2f}"
                if abs(val_c - max_class) < 1e-9: s_c = f"\\textbf{{{s_c}}}"
                
                row_values.append(f"{s_r} / {s_c}")
        
        line = f"{backbone_display} & {pooling} & " + " & ".join(row_values) + " \\\\"
        latex_str.append(line)

    latex_str.append("\\bottomrule")
    latex_str.append("\\end{tabular}")
    latex_str.append("\\end{table*}")

    return "\n".join(latex_str)

print(generate_latex_table_val_selected())

\begin{table*}[ht]
\centering
\small
\caption{Test Accuracy (Rank / Class) selected at best Validation Epoch.}
\label{tab:backbone_results}
\begin{tabular}{llccccc}
\toprule
 & & \multicolumn{5}{c}{Unfreeze Depth (Blocks)} \\
\cmidrule(lr){3-7}
Backbone & Pooling & No Finetune & 1 & 4 & 8 & 12 \\
\midrule
\midrule
\textbf{DINOv3} & CLS & 71.40 / 68.24 & 75.51 / 74.32 & \textbf{76.80} / \textbf{76.20} & 75.77 / 75.60 & 75.43 / 74.23 \\
 & Concat & 75.00 / 74.06 & 75.86 / 75.68 & 73.20 / 74.83 & 76.37 / \textbf{76.63} & \textbf{76.97} / 75.68 \\
 & Top-K & 69.26 / 57.45 & 73.46 / 73.54 & \textbf{76.37} / \textbf{76.37} & 73.54 / 73.63 & 74.49 / 72.95 \\
\midrule
\textbf{DeiT III} & CLS & 69.09 / 66.70 & 73.03 / 73.63 & 75.17 / \textbf{74.74} & 75.17 / 73.12 & \textbf{75.60} / 73.54 \\
 & Concat & 72.00 / 70.12 & 73.72 / 73.46 & 74.06 / 72.35 & 74.57 / 74.14 & \textbf{74.83} / \textbf{74.40} \\
 & Top-K & 69.61 / 67.64 & \textbf{73.54} / 69.09 & 72.43 / \textbf{73.29} & 72.95 / 69.09 & 71